In [1]:
#Code taken from https://www.lattices.io/algorithms/lll
import numpy as np

def gram_schmidt(B):
    """
    Compute Gram-Schmidt orthogonalization.

    Returns:
        B_star: orthogonal vectors (same shape as B)
        mu: matrix of Gram-Schmidt coefficients
             mu[i,j] = <b_i, b*_j> / <b*_j, b*_j>
    """
    n = B.shape[0]
    B_star = np.zeros_like(B, dtype=float)
    mu = np.zeros((n, n), dtype=float)

    for i in range(n):
        B_star[i] = B[i].copy()
        for j in range(i):
            # project b_i onto each previous orthogonal vector b*_j
            mu[i, j] = np.dot(B[i], B_star[j]) / np.dot(B_star[j], B_star[j])
            B_star[i] -= mu[i, j] * B_star[j]

    return B_star, mu


def lll_reduction(B, delta=0.75):
    """
    LLL lattice basis reduction.

    Args:
        B: integer matrix where rows are basis vectors (n x m)
        delta: reduction parameter in (0.25, 1). Default 0.75.
               Higher delta → better output but more iterations.

    Returns:
        B: LLL-reduced basis (modified in place)
    """
    B = np.array(B, dtype=float)
    n = B.shape[0]
    B_star, mu = gram_schmidt(B)

    k = 1
    while k < n:
        # === Step 1: Size reduction ===
        # Make |mu[k,j]| <= 0.5 for all j < k
        # This ensures basis vectors are not too skewed
        for j in range(k - 1, -1, -1):
            if abs(mu[k, j]) > 0.5:
                # Subtract the nearest integer multiple of b_j from b_k
                q = round(mu[k, j])
                B[k] -= q * B[j]
                # Update Gram-Schmidt coefficients
                B_star, mu = gram_schmidt(B)

        # === Step 2: Lovász condition ===
        # Check: delta * ||b*_{k-1}||^2 <= ||b*_k||^2 + mu[k,k-1]^2 * ||b*_{k-1}||^2
        # This prevents Gram-Schmidt norms from dropping too fast
        norm_k = np.dot(B_star[k], B_star[k])
        norm_k1 = np.dot(B_star[k - 1], B_star[k - 1])
        lovasz = norm_k + mu[k, k - 1] ** 2 * norm_k1

        if lovasz >= delta * norm_k1:
            # Condition satisfied → move to next index
            k += 1
        else:
            # Condition violated → swap b_k and b_{k-1}
            B[[k, k - 1]] = B[[k - 1, k]]
            B_star, mu = gram_schmidt(B)
            # Step back (but not below index 1)
            k = max(k - 1, 1)

    return B


# === Example usage ===
# Create a "bad" 4-dimensional basis with nearly parallel vectors
basis = np.array([
    [1,  1,  1,  1],
    [-1, 0,  0,  0],
    [3,  5,  6,  6],
    [1,  2,  3,  7],
])

print("Original basis:")
print(basis)
print("Norms:", [round(np.linalg.norm(row), 2) for row in basis])

reduced = lll_reduction(basis, delta=0.75)
print("\nLLL-reduced basis:")
print(reduced.astype(int))
print("Norms:", [round(np.linalg.norm(row), 2) for row in reduced])

Original basis:
[[ 1  1  1  1]
 [-1  0  0  0]
 [ 3  5  6  6]
 [ 1  2  3  7]]
Norms: [2.0, 1.0, 10.3, 7.94]

LLL-reduced basis:
[[-1  0  0  0]
 [ 0 -1  0  0]
 [ 0  0  1  1]
 [ 0  0 -2  2]]
Norms: [1.0, 1.0, 1.41, 2.83]
